In [1]:
import duckdb

In [2]:
con = duckdb.connect(database='dadosduckdb.db',read_only=False)

In [3]:
df = con.execute("""select * from (
                    select *,
                    row_number() over (partition by natbr order by data_ingestao desc) as row
                    from bronze_produtos
                    where data_ingestao >= '2026-04-01'
                 ) where row = 1
                 """).fetchdf()
print(df)

   NATBR     MAKTX WERKS MAINS LABST NOME_ARQUIVO              DATA_INGESTAO  \
0  10001  PARAFUSO  BT10   100   100  z0019_1.csv 2026-04-02 05:56:21.869342   
1  10002   MARTELO  BT50   100  1500  z0019_1.csv 2026-04-02 05:56:21.869342   
2  10004     SERRA  BT50   100   200  z0019_2.csv 2026-04-02 05:56:42.413375   
3  10005   MACHADO  BT10   100   100  z0019_2.csv 2026-04-02 05:56:42.413375   
4  10003     PREGO  BT10   100    60  z0019_2.csv 2026-04-02 05:56:42.413375   

   row  
0    1  
1    1  
2    1  
3    1  
4    1  


In [4]:
df_final = df.drop(columns=['NOME_ARQUIVO','DATA_INGESTAO','row'])
df_final = df_final.rename(columns={'NATBR':'id'})
df_final = df_final.rename(columns={'MAKTX':'nome'})
df_final = df_final.rename(columns={'WERKS':'categoria'})
df_final = df_final.rename(columns={'MAINS':'fornecedor'})
df_final = df_final.rename(columns={'LABST':'preco'})

print(df_final)

      id      nome categoria fornecedor preco
0  10001  PARAFUSO      BT10        100   100
1  10002   MARTELO      BT50        100  1500
2  10004     SERRA      BT50        100   200
3  10005   MACHADO      BT10        100   100
4  10003     PREGO      BT10        100    60


In [5]:
df2 = df_final
df2 = df2.astype(
        {
            'id': 'int32',
            'nome': 'string',
            'categoria': 'string',
            'fornecedor': 'int32',
            'preco': 'float32'
        }
    )
print(df2.dtypes)

id              int32
nome           string
categoria      string
fornecedor      int32
preco         float32
dtype: object


In [6]:
con.execute(
    """
        create table if not exists produtos (
            id bigint,
            nome Text,
            categoria text,
            fornecedor bigint,
            preco float
        )
    """
)


In [7]:
con.execute('insert into produtos select * from df2')

In [8]:
df_resultado = con.execute('select * from produtos').fetchdf()
print(df_resultado.head())

      id      nome categoria  fornecedor   preco
0  10001  PARAFUSO      BT10         100   100.0
1  10002   MARTELO      BT50         100  1500.0
2  10004     SERRA      BT50         100   200.0
3  10005   MACHADO      BT10         100   100.0
4  10003     PREGO      BT10         100    60.0


In [9]:
con.close()